In [98]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [99]:
import os, h5py, cv2, numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


In [100]:
TRAIN_H5 = "/content/drive/MyDrive/morph_project/processed/train.h5"
TEST_H5  = "/content/drive/MyDrive/morph_project/processed/test.h5"

OUT_TRAIN_FEATURES = "/content/drive/MyDrive/morph_project/fusion/train_features.h5"
OUT_TEST_FEATURES  = "/content/drive/MyDrive/morph_project/fusion/test_features.h5"

FUSION_MODEL_PATH = "/content/drive/MyDrive/morph_project/fusion/fusion2_feature_level_best.pth"

print(os.path.exists(TRAIN_H5), os.path.exists(TEST_H5))


True True


In [101]:
vit  = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=0)
eff  = timm.create_model("efficientnet_b3", pretrained=False, num_classes=0)
conv = timm.create_model("convnextv2_tiny", pretrained=False, num_classes=0)
swin = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=0)

vit.load_state_dict(torch.load("/content/drive/MyDrive/morph_project/models/vit_advanced.pth", map_location=device), strict=False)
eff.load_state_dict(torch.load("/content/drive/MyDrive/morph_project/models/efficientnet_b3_tuned_baseline.pth", map_location=device), strict=False)
conv.load_state_dict(torch.load("/content/drive/MyDrive/morph_project/models/convnextv2_best_224_finetuned.pth", map_location=device), strict=False)
swin.load_state_dict(torch.load("/content/drive/MyDrive/morph_project/swin224_recall_tuned.pth", map_location=device), strict=False)

models = [vit, eff, conv, swin]

for m in models:
    m.to(device)
    m.eval()
    for p in m.parameters():
        p.requires_grad = False

@torch.no_grad()
def extract_features(x):
    return torch.cat([vit(x), eff(x), conv(x), swin(x)], dim=1)


In [102]:
def create_feature_h5(IN_H5, OUT_H5, batch_size=32):
    with h5py.File(IN_H5, "r") as fin, h5py.File(OUT_H5, "w") as fout:
        X = fin["X"]
        y = fin["y"]
        n = X.shape[0]

        sample = torch.tensor(X[0:1]).permute(0,3,1,2).float().to(device)
        feat_dim = extract_features(sample).shape[1]

        fout.create_dataset("X", shape=(n, feat_dim), dtype=np.float32)
        fout.create_dataset("y", data=y, dtype=np.int64)

        idx = 0
        for i in tqdm(range(0, n, batch_size), desc=f"Extracting {OUT_H5}"):
            xb = torch.tensor(X[i:i+batch_size]).permute(0,3,1,2).float().to(device)
            feats = extract_features(xb).cpu().numpy()
            fout["X"][idx:idx+feats.shape[0]] = feats
            idx += feats.shape[0]

    print("Saved:", OUT_H5)


In [103]:
create_feature_h5(TRAIN_H5, OUT_TRAIN_FEATURES)
create_feature_h5(TEST_H5, OUT_TEST_FEATURES)


Extracting /content/drive/MyDrive/morph_project/fusion/train_features.h5: 100%|██████████| 225/225 [03:13<00:00,  1.16it/s]


Saved: /content/drive/MyDrive/morph_project/fusion/train_features.h5


Extracting /content/drive/MyDrive/morph_project/fusion/test_features.h5: 100%|██████████| 57/57 [00:51<00:00,  1.11it/s]

Saved: /content/drive/MyDrive/morph_project/fusion/test_features.h5


In [104]:
class FeatureH5Dataset(Dataset):
    def __init__(self, path):
        self.path = path
        with h5py.File(path, "r") as f:
            self.n = f["X"].shape[0]

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        with h5py.File(self.path, "r") as f:
            x = f["X"][idx]
            y = f["y"][idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)


In [105]:
train_loader = DataLoader(
    FeatureH5Dataset(OUT_TRAIN_FEATURES),
    batch_size=64, shuffle=True, num_workers=2
)

test_loader = DataLoader(
    FeatureH5Dataset(OUT_TEST_FEATURES),
    batch_size=64, shuffle=False, num_workers=2
)

X, _ = next(iter(train_loader))
feat_dim = X.shape[1]
print("Feature dim:", feat_dim)


Feature dim: 3840


In [106]:
class FusionHead(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, 2)
        )

    def forward(self, x):
        return self.net(x)

model = FusionHead(feat_dim).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)


In [107]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss {total_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), FUSION_MODEL_PATH)
print("Fusion-2 model saved")


Epoch 1/10 | Loss 0.1186
Epoch 2/10 | Loss 0.0542
Epoch 3/10 | Loss 0.0442
Epoch 4/10 | Loss 0.0388
Epoch 5/10 | Loss 0.0343
Epoch 6/10 | Loss 0.0357
Epoch 7/10 | Loss 0.0318
Epoch 8/10 | Loss 0.0275
Epoch 9/10 | Loss 0.0262
Epoch 10/10 | Loss 0.0304
Fusion-2 model saved


In [108]:
model.eval()
yt, yp, ys = [], [], []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        probs = torch.softmax(model(X), dim=1)
        yt.extend(y.numpy())
        yp.extend(probs.argmax(1).cpu().numpy())
        ys.extend(probs[:,1].cpu().numpy())

print("Accuracy :", accuracy_score(yt, yp))
print("Precision:", precision_score(yt, yp))
print("Recall   :", recall_score(yt, yp))
print("F1-score :", f1_score(yt, yp))
print("ROC-AUC  :", roc_auc_score(yt, ys))
print("Confusion Matrix:\n", confusion_matrix(yt, yp))


Accuracy : 0.8624514697726012
Precision: 0.6398305084745762
Recall   : 0.48089171974522293
F1-score : 0.5490909090909091
ROC-AUC  : 0.7940470028617506
Confusion Matrix:
 [[1404   85]
 [ 163  151]]


In [110]:
from google.colab import files

ILLICIT_THRESHOLD = 0.30   # ← LOWER THAN 0.5 (IMPORTANT)

def preprocess_image(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224,224))
    img = img.astype(np.float32) / 255.0
    img = torch.tensor(img).permute(2,0,1).unsqueeze(0).to(device)
    return img

@torch.no_grad()
def predict_image(path):
    x = preprocess_image(path)
    feats = extract_features(x)
    logits = model(feats)[0]
    probs = torch.softmax(logits, dim=0)

    print("\n---------------------------")
    print("Authentic prob:", float(probs[0]))
    print("Tampered  prob:", float(probs[1]))

    pred = int(probs[1] >= ILLICIT_THRESHOLD)
    print("Prediction:", "Tampered" if pred else "Authentic")
    print("---------------------------\n")

uploaded = files.upload()
for name in uploaded:
    predict_image(name)


Saving casias1.png to casias1 (2).png

---------------------------
Authentic prob: 0.9584880471229553
Tampered  prob: 0.04151191562414169
Prediction: Authentic
---------------------------

